# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [4]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [5]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)

In [6]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [7]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [8]:
from pyspark.sql.functions import monotonically_increasing_id
# Add a unique ID to each trip
df_with_id = df_trips.withColumn("trip_id", monotonically_increasing_id())


In [9]:
from pyspark.sql.functions import col
# Show the trip with the most passengers
df_with_id.orderBy(col("passenger_count").desc()).show(1)


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       2| 2019-01-05 13:12:29|  2019-01-05 13:12:32|            9.0|          0.0|       5.0|                 N|          68|          68|           1

In [10]:
from pyspark.sql.functions import avg
# Show average passenger count
df_with_id.select(avg("passenger_count")).show()


+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [11]:
from pyspark.sql.functions import min, max
# Show minimum and maximum distance
df_with_id.select(min("trip_distance"), max("trip_distance")).show()


+------------------+------------------+
|min(trip_distance)|max(trip_distance)|
+------------------+------------------+
|               0.0|             831.8|
+------------------+------------------+



In [12]:
from pyspark.sql.functions import col, unix_timestamp, min, max

# Add trip duration in minutes
df_with_duration = df_with_id.withColumn(
    "duration_minutes",
    # Dropoff minus pickup, converted from seconds to minutes
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
)

# Show minimum and maximum duration
df_with_duration.select(min("duration_minutes"), max("duration_minutes")).show()


+---------------------+---------------------+
|min(duration_minutes)|max(duration_minutes)|
+---------------------+---------------------+
|             -84280.5|    43648.01666666667|
+---------------------+---------------------+



In [13]:
# Filter trips by duration and distance
df_clean = df_with_duration.filter(
    # Keep positive durations
    (col("duration_minutes") > 0) & 
    # Keep trips up to 12 hours
    (col("duration_minutes") <= 720) &
    # Keep positive distances
    (col("trip_distance") > 0)
)
# Show minimum and maximum duration
df_clean.select(min("duration_minutes"), max("duration_minutes")).show()


+---------------------+---------------------+
|min(duration_minutes)|max(duration_minutes)|
+---------------------+---------------------+
| 0.016666666666666666|                719.8|
+---------------------+---------------------+



In [14]:
from pyspark.sql.functions import to_date, count, col
# Extract pickup date, group by date, and count trips
daily_trips = df_clean.withColumn("pickup_date", to_date("tpep_pickup_datetime")) \
    .groupBy("pickup_date") \
    .agg(count("*").alias("trip_count"))
# Display daily trip counts
daily_trips.show()


+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-02-23|         1|
| 2009-01-01|        40|
| 2019-01-07|    226697|
| 2019-01-08|    235035|
| 2019-01-28|    238900|
| 2019-03-17|         2|
| 2019-04-28|         3|
| 2019-01-30|    274377|
| 2019-01-26|    269360|
| 2019-06-10|         2|
| 2018-12-30|         7|
| 2019-02-01|        61|
| 2019-05-20|         1|
| 2019-01-20|    201019|
| 2019-03-19|         3|
| 2019-01-22|    252614|
| 2019-01-19|    234091|
| 2019-07-23|         1|
| 2019-01-05|    234224|
| 2019-01-03|    221652|
+-----------+----------+
only showing top 20 rows


In [15]:
# Show the busiest date
daily_trips.orderBy(col("trip_count").desc()).show(1)
# Show the slowest date
daily_trips.orderBy(col("trip_count").asc()).show(1)


+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-01-25|    289786|
+-----------+----------+
only showing top 1 row
+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-05-20|         1|
+-----------+----------+
only showing top 1 row


In [16]:
from pyspark.sql.functions import hour, count, col

hourly_trips = (
    df_clean
    # Extract pickup hour (0-23)
    .withColumn("pickup_hour", hour("tpep_pickup_datetime"))
    # Group by pickup_hour
    .groupBy("pickup_hour")
    # Count trips in each group
    .agg(count("*").alias("trip_count"))
)

print("Busiest hour:")
# Show the busiest hour
hourly_trips.orderBy(col("trip_count").desc()).show(1)

print("Slowest hour:")
# Show the slowest hour
hourly_trips.orderBy(col("trip_count").asc()).show(1)

# Show all 24 hours in order
hourly_trips.orderBy("pickup_hour").show(24)


Busiest hour:
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|         18|    511222|
+-----------+----------+
only showing top 1 row
Slowest hour:
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|          4|     60037|
+-----------+----------+
only showing top 1 row
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|          0|    205293|
|          1|    147304|
|          2|    107788|
|          3|     76657|
|          4|     60037|
|          5|     74058|
|          6|    176054|
|          7|    301420|
|          8|    370460|
|          9|    362758|
|         10|    358096|
|         11|    371961|
|         12|    397465|
|         13|    400257|
|         14|    428840|
|         15|    448050|
|         16|    416248|
|         17|    464140|
|         18|    511222|
|         19|    471619|
|         20|    419869|
|         21|    406522|
|         22|    365938|
|         23|    279187|
+

In [17]:
from pyspark.sql.functions import to_date, date_format, dayofweek, count, avg, col

daily_counts = (
    df_clean
    # Extract pickup date
    .withColumn("trip_date", to_date("tpep_pickup_datetime"))
    # Group by trip_date
    .groupBy("trip_date")
    # Count trips in each group
    .agg(count("*").alias("daily_total"))
)

avg_by_day = (
    daily_counts
    # Add weekday name
    .withColumn("day_of_week", date_format("trip_date", "EEEE"))
    # Add weekday number (Sunday = 1, Saturday = 7)
    .withColumn("day_num", dayofweek("trip_date"))
    # Group by day_of_week, day_num
    .groupBy("day_of_week", "day_num")
    # Average daily trip counts
    .agg(avg("daily_total").alias("avg_daily_trips"))
)

print("Busiest day of the week on average:")
# Show the busiest weekday on average
avg_by_day.orderBy(col("avg_daily_trips").desc()).select("day_of_week", "avg_daily_trips").show(1)

print("Slowest day of the week on average:")
# Show the slowest weekday on average
avg_by_day.orderBy(col("avg_daily_trips").asc()).select("day_of_week", "avg_daily_trips").show(1)

# Show averages in weekday order
avg_by_day.orderBy("day_num").select("day_of_week", "avg_daily_trips").show()


Busiest day of the week on average:
+-----------+------------------+
|day_of_week|   avg_daily_trips|
+-----------+------------------+
|   Thursday|224034.83333333334|
+-----------+------------------+
only showing top 1 row
Slowest day of the week on average:
+-----------+---------------+
|day_of_week|avg_daily_trips|
+-----------+---------------+
|     Monday|        89915.9|
+-----------+---------------+
only showing top 1 row
+-----------+------------------+
|day_of_week|   avg_daily_trips|
+-----------+------------------+
|     Sunday|         106354.75|
|     Monday|           89915.9|
|    Tuesday|          119691.0|
|  Wednesday|179043.57142857142|
|   Thursday|224034.83333333334|
|     Friday|          215330.8|
|   Saturday|142881.14285714287|
+-----------+------------------+



In [18]:
from pyspark.sql.functions import corr, col

# Select trips for the tip analysis
df_tips = df_clean.filter(
    # Keep card payments
    (col("payment_type") == 1) & 
    # Keep non-negative tips
    (col("tip_amount") >= 0) & 
    # Keep positive passenger counts
    (col("passenger_count") > 0)
)

# Pearson correlation: tip amount vs distance
corr_distance = df_tips.stat.corr("tip_amount", "trip_distance")
# Pearson correlation: tip amount vs passenger count
corr_passengers = df_tips.stat.corr("tip_amount", "passenger_count")

# Print correlation to 4 decimal places
print(f"Correlation between Tip Amount and Trip Distance: {corr_distance:.4f}")
# Print correlation to 4 decimal places
print(f"Correlation between Tip Amount and Passenger Count: {corr_passengers:.4f}")


Correlation between Tip Amount and Trip Distance: 0.7022
Correlation between Tip Amount and Passenger Count: 0.0107


In [19]:
from pyspark.sql.functions import col

(
    df_clean
    # Keep trip ID, amounts, and pickup time
    .select("trip_id", "extra", "fare_amount", "total_amount", "tpep_pickup_datetime")
    # Sort extra charges from highest to lowest
    .orderBy(col("extra").desc())
    # Show 5 trips without truncating values
    .show(5, truncate=False)
)


+-----------+-----+-----------+------------+--------------------+
|trip_id    |extra|fare_amount|total_amount|tpep_pickup_datetime|
+-----------+-----+-----------+------------+--------------------+
|94489591564|18.5 |52.0       |88.3        |2019-01-02 16:33:28 |
|94491735598|18.5 |49.0       |96.36       |2019-01-11 16:08:48 |
|94489415061|18.5 |39.5       |70.8        |2019-01-01 16:09:32 |
|94489823715|18.5 |61.0       |92.3        |2019-01-03 18:32:36 |
|94489828820|18.5 |47.5       |94.56       |2019-01-03 18:19:33 |
+-----------+-----+-----------+------------+--------------------+
only showing top 5 rows


### Data Anomalies and Outliers

* **Negative Durations (e.g., -84,280 min):** Dropoff timestamp occurs before pickup timestamp, caused by clock desynchronization or meter software bugs.
* **Extreme Durations (> 40,000 min / ~30 days):** Drivers forgot to turn off the taximeter, leaving the session open until month-end.
* **Negative Fares & Extras (`total_amount < 0`):** Represent refunds, cancellations, or transaction disputes rather than real rides.
* **Zero Distance with Positive Fare (`trip_distance = 0`):** Trips cancelled on arrival, passenger wait times, or flat-rate charges with GPS inactive.
* **Extreme `extra` Charges:** Standard NYC TLC extra surcharges are strictly $0.50 (night) or $1.00 (rush hour). High values indicate manual overrides or data-entry errors.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

I answered Part 2 using **pyspark** (same choice as Part 1). Every borough
question relies on joining the trips with the taxi zone lookup table, and I
reuse the cleaned `df_clean` dataframe built in Part 1.

**Loading the taxi zone lookup table**

In [20]:
# Taxi zone lookup URL
zone_url = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
# Download the file
response = requests.get(zone_url)

# Local file name
zone_lookup_file = "taxi_zone_lookup.csv"
# Save only if the request succeeded
if response.status_code == 200:
    # Open the file in binary write mode
    with open(zone_lookup_file, "wb") as f:
        # Write downloaded content to disk
        f.write(response.content)


In [21]:
# Read CSV with column headers and inferred data types
df_zones = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(zone_lookup_file)

# Preview 5 zones
df_zones.show(5)


+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


**Attaching a borough to every trip.**
I join `df_clean` with the lookup twice: once on `PULocationID` to get the
pickup borough, once on `DOLocationID` to get the dropoff borough.

In [22]:
# Join pickup zones to boroughs, keeping all trips
trips_pu = df_clean.join(
    df_zones, df_clean.PULocationID == df_zones.LocationID, "left"
# Rename the borough column and drop unused lookup fields
).withColumnRenamed("Borough", "pickup_borough").drop("LocationID", "Zone", "service_zone")

# Join dropoff zones to boroughs, keeping all trips
trips_do = df_clean.join(
    df_zones, df_clean.DOLocationID == df_zones.LocationID, "left"
# Rename the borough column and drop unused lookup fields
).withColumnRenamed("Borough", "dropoff_borough").drop("LocationID", "Zone", "service_zone")


### Which borough had the most pickups? the most dropoffs?

In [23]:
from pyspark.sql.functions import count, col

print("Pickups by borough:")
# Count pickups per borough and sort from most to least
trips_pu.groupBy("pickup_borough") \
    .agg(count("*").alias("pickup_count")) \
    .orderBy(col("pickup_count").desc()) \
    .show()

print("Dropoffs by borough:")
# Count dropoffs per borough and sort from most to least
trips_do.groupBy("dropoff_borough") \
    .agg(count("*").alias("dropoff_count")) \
    .orderBy(col("dropoff_count").desc()) \
    .show()


Pickups by borough:
+--------------+------------+
|pickup_borough|pickup_count|
+--------------+------------+
|     Manhattan|     6904024|
|        Queens|      456491|
|       Unknown|      151323|
|      Brooklyn|       89450|
|         Bronx|       17260|
|           N/A|        2217|
| Staten Island|         327|
|           EWR|         151|
+--------------+------------+

Dropoffs by borough:
+---------------+-------------+
|dropoff_borough|dropoff_count|
+---------------+-------------+
|      Manhattan|      6772482|
|         Queens|       326355|
|       Brooklyn|       297323|
|        Unknown|       140704|
|          Bronx|        56998|
|            N/A|        14792|
|            EWR|        10450|
|  Staten Island|         2139|
+---------------+-------------+



### Busy / slow times by borough
I count trips per (borough, hour), then use a window ranked by trip count to
keep only the busiest and the slowest hour of each borough.

In [24]:
from pyspark.sql.functions import hour, row_number
# Import window functions
from pyspark.sql.window import Window

# Extract pickup hour and count trips per borough and hour
hourly_borough = trips_pu.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
    .groupBy("pickup_borough", "pickup_hour") \
    .agg(count("*").alias("trip_count"))

# Rank hours from busiest to slowest within each borough
w_desc = Window.partitionBy("pickup_borough").orderBy(col("trip_count").desc())
# Rank hours from slowest to busiest within each borough
w_asc = Window.partitionBy("pickup_borough").orderBy(col("trip_count").asc())

print("Busiest hour by borough:")
# Keep rank 1 per borough, drop the rank, and display busiest hours
hourly_borough.withColumn("rk", row_number().over(w_desc)) \
    .filter(col("rk") == 1).drop("rk") \
    .orderBy(col("trip_count").desc()).show()

print("Slowest hour by borough:")
# Keep rank 1 per borough, drop the rank, and display slowest hours
hourly_borough.withColumn("rk", row_number().over(w_asc)) \
    .filter(col("rk") == 1).drop("rk") \
    .orderBy(col("trip_count").asc()).show()


Busiest hour by borough:
+--------------+-----------+----------+
|pickup_borough|pickup_hour|trip_count|
+--------------+-----------+----------+
|     Manhattan|         18|    468744|
|        Queens|         21|     28903|
|       Unknown|         18|     10300|
|      Brooklyn|          8|      6803|
|         Bronx|          7|      1727|
|           N/A|         14|       137|
| Staten Island|          8|        36|
|           EWR|         15|        23|
+--------------+-----------+----------+

Slowest hour by borough:
+--------------+-----------+----------+
|pickup_borough|pickup_hour|trip_count|
+--------------+-----------+----------+
|           EWR|          4|         1|
| Staten Island|         22|         2|
|           N/A|          4|        46|
|         Bronx|          3|       201|
|       Unknown|          4|      1214|
|      Brooklyn|          3|      1841|
|        Queens|          3|      2914|
|     Manhattan|          4|     52738|
+--------------+-----------+-

### Busiest day of the week by borough
Same logic as Part 1: count per calendar date first, then average those daily
totals per weekday so a weekday that appears 5 times in the month is not
favoured over one that appears 4 times.

In [25]:
from pyspark.sql.functions import to_date, date_format, dayofweek, avg

# Extract pickup date and count trips per borough and date
daily_borough = trips_pu.withColumn("trip_date", to_date("tpep_pickup_datetime")) \
    .groupBy("pickup_borough", "trip_date") \
    .agg(count("*").alias("daily_total"))

# Add weekday name and number, then average daily counts per borough
avg_day_borough = daily_borough \
    .withColumn("day_of_week", date_format("trip_date", "EEEE")) \
    .withColumn("day_num", dayofweek("trip_date")) \
    .groupBy("pickup_borough", "day_of_week", "day_num") \
    .agg(avg("daily_total").alias("avg_daily_trips"))

# Rank weekdays by average trip count within each borough
w_desc_day = Window.partitionBy("pickup_borough").orderBy(col("avg_daily_trips").desc())
print("Busiest weekday by borough (on average):")
# Keep the busiest weekday per borough and drop ranking fields
avg_day_borough.withColumn("rk", row_number().over(w_desc_day)) \
    .filter(col("rk") == 1).drop("rk", "day_num") \
    .orderBy(col("avg_daily_trips").desc()).show()


Busiest weekday by borough (on average):
+--------------+-----------+------------------+
|pickup_borough|day_of_week|   avg_daily_trips|
+--------------+-----------+------------------+
|     Manhattan|   Thursday|203592.16666666666|
|        Queens|    Tuesday|           15254.2|
|       Unknown|   Thursday|            5505.4|
|      Brooklyn|     Friday|           3189.75|
|         Bronx|     Friday|            640.75|
|           N/A|    Tuesday|              77.8|
| Staten Island|     Friday|              15.5|
|           EWR|     Friday|              8.75|
+--------------+-----------+------------------+



### Average trip distance and average fare by borough

In [26]:
# Import Spark rounding as sround
from pyspark.sql.functions import round as sround

print("Average trip distance by borough (miles):")
# Average distance in miles per borough, round to 2 decimals, and sort descending
trips_pu.groupBy("pickup_borough") \
    .agg(sround(avg("trip_distance"), 2).alias("avg_distance_mi")) \
    .orderBy(col("avg_distance_mi").desc()).show()

print("Average fare by borough ($):")
# Average fare per borough, round to 2 decimals, and sort descending
trips_pu.groupBy("pickup_borough") \
    .agg(sround(avg("fare_amount"), 2).alias("avg_fare")) \
    .orderBy(col("avg_fare").desc()).show()


Average trip distance by borough (miles):
+--------------+---------------+
|pickup_borough|avg_distance_mi|
+--------------+---------------+
| Staten Island|          13.75|
|        Queens|          11.59|
|           EWR|            7.8|
|         Bronx|           7.56|
|           N/A|            5.6|
|      Brooklyn|           4.91|
|       Unknown|           2.54|
|     Manhattan|           2.24|
+--------------+---------------+

Average fare by borough ($):
+--------------+--------+
|pickup_borough|avg_fare|
+--------------+--------+
|           EWR|   70.58|
| Staten Island|   43.93|
|           N/A|   38.04|
|        Queens|   35.61|
|         Bronx|   26.74|
|      Brooklyn|   18.77|
|       Unknown|   11.74|
|     Manhattan|    10.7|
+--------------+--------+



### Highest / lowest fare for a single trip, and its borough
I keep only strictly positive fares for the "lowest" question, because negative
values are refunds / disputes (see the outlier notes at the end of Part 1).

In [27]:
# Keep positive fares and select trip details
fares = trips_pu.filter(col("fare_amount") > 0) \
    .select("trip_id", "fare_amount", "pickup_borough", "tpep_pickup_datetime")

print("Highest fare trip:")
# Show the highest fare trip
fares.orderBy(col("fare_amount").desc()).show(1, truncate=False)

print("Lowest (positive) fare trip:")
# Show the lowest positive fare trip
fares.orderBy(col("fare_amount").asc()).show(1, truncate=False)


Highest fare trip:
+-----------+-----------+--------------+--------------------+
|trip_id    |fare_amount|pickup_borough|tpep_pickup_datetime|
+-----------+-----------+--------------+--------------------+
|94491780167|623259.86  |Manhattan     |2019-01-11 19:33:15 |
+-----------+-----------+--------------+--------------------+
only showing top 1 row
Lowest (positive) fare trip:
+-----------+-----------+--------------+--------------------+
|trip_id    |fare_amount|pickup_borough|tpep_pickup_datetime|
+-----------+-----------+--------------+--------------------+
|94489290779|0.01       |Manhattan     |2019-01-01 00:53:53 |
+-----------+-----------+--------------+--------------------+
only showing top 1 row


### Comparing January 2019 with the most recent January (2025)
I load the most recently available january (2025), apply the *exact same*
cleaning rules as in Part 1 so the comparison is fair, and put the key averages
side by side.

In [28]:
# January 2025 trip data URL
download_url_2025 = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
# Download the file
response = requests.get(download_url_2025)

# Local file name
jan_2025_trip_data = "yellow_tripdata_2025-01.parquet"
# Save only if the request succeeded
if response.status_code == 200:
    # Open the file in binary write mode
    with open(jan_2025_trip_data, "wb") as f:
        # Write downloaded content to disk
        f.write(response.content)

# Load the Parquet file into a Spark DataFrame
df_trips_2025 = spark.read.parquet(jan_2025_trip_data)


In [29]:
# Calculate duration and filter 2025 trips
df_2025_clean = df_trips_2025.withColumn(
    "duration_minutes",
    # Dropoff minus pickup, converted from seconds to minutes
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
).filter(
    # Keep positive durations
    (col("duration_minutes") > 0) &
    # Keep trips up to 12 hours
    (col("duration_minutes") <= 720) &
    # Keep positive distances
    (col("trip_distance") > 0)
)


In [30]:
from pyspark.sql.functions import lit

# Build one row of averages for a dataset
def avg_metrics(df, label):
    # Calculate averages rounded to 3 decimal places
    return df.select(
        # Add the dataset label
        lit(label).alias("dataset"),
        # Average passenger count
        sround(avg("passenger_count"), 3).alias("avg_passengers"),
        # Average distance in miles
        sround(avg("trip_distance"), 3).alias("avg_distance_mi"),
        # Average fare
        sround(avg("fare_amount"), 3).alias("avg_fare"),
        # Average tip
        sround(avg("tip_amount"), 3).alias("avg_tip"),
        # Average total amount
        sround(avg("total_amount"), 3).alias("avg_total"),
        # Average duration in minutes
        sround(avg("duration_minutes"), 3).alias("avg_duration_min"),
    )

# Stack the 2019 and 2025 summary rows
comparison = avg_metrics(df_clean, "Jan 2019").union(avg_metrics(df_2025_clean, "Jan 2025"))
# Display both summary rows
comparison.show(truncate=False)


+--------+--------------+---------------+--------+-------+---------+----------------+
|dataset |avg_passengers|avg_distance_mi|avg_fare|avg_tip|avg_total|avg_duration_min|
+--------+--------------+---------------+--------+-------+---------+----------------+
|Jan 2019|1.568         |2.847          |12.357  |1.813  |15.629   |13.034          |
|Jan 2025|1.297         |6.014          |17.066  |3.003  |25.67    |14.716          |
+--------+--------------+---------------+--------+-------+---------+----------------+



### Observations on 2019 vs 2025
The table above compares the core averages on identically-cleaned data. Things
to keep in mind when reading it:

- **Fares / totals**: the NYC TLC raised the base yellow-cab rates at the end of
  2022, and congestion pricing started in January 2025, so the average fare and
  the average total are expected to be clearly higher in 2025 than in 2019.
- **Passenger count**: ridership patterns changed after COVID, so the average
  `passenger_count` tends to be flat or slightly lower.
- **Trip distance / duration**: broadly comparable, any change reflecting demand
  and traffic rather than a schema difference (the yellow-taxi schema is stable).

The two rows produced above give the exact magnitude of each change.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

**SQL setup**


In [31]:
# Make the DataFrames available to Spark SQL
df_with_id.createOrReplaceTempView("trips_sql")
df_clean.createOrReplaceTempView("clean_trips_sql")
df_zones.createOrReplaceTempView("zones_sql")


**1. What is the average passenger count?**


In [32]:
spark.sql("""
    -- Average passenger count across all trips
    SELECT AVG(passenger_count) AS avg_passengers
    FROM trips_sql
""").show()


+------------------+
|    avg_passengers|
+------------------+
|1.5670317144945614|
+------------------+



**2. Which day had the most trips?**


In [33]:
spark.sql("""
    -- Count trips for each pickup date
    SELECT TO_DATE(tpep_pickup_datetime) AS pickup_date,
           COUNT(*) AS trip_count
    FROM clean_trips_sql
    GROUP BY TO_DATE(tpep_pickup_datetime)
    -- Keep the date with the most trips
    ORDER BY trip_count DESC
    LIMIT 1
""").show()


+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-01-25|    289786|
+-----------+----------+



**3. Which borough had the most pickups?**


In [34]:
spark.sql("""
    -- Count pickups per borough
    SELECT z.Borough AS pickup_borough,
           COUNT(*) AS pickup_count
    FROM clean_trips_sql AS t
    -- Match pickup zones to boroughs, keeping all trips
    LEFT JOIN zones_sql AS z
        ON t.PULocationID = z.LocationID
    GROUP BY z.Borough
    -- Show the busiest borough first
    ORDER BY pickup_count DESC
""").show()


+--------------+------------+
|pickup_borough|pickup_count|
+--------------+------------+
|     Manhattan|     6904024|
|        Queens|      456491|
|       Unknown|      151323|
|      Brooklyn|       89450|
|         Bronx|       17260|
|           N/A|        2217|
| Staten Island|         327|
|           EWR|         151|
+--------------+------------+



# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing